In [6]:
from langchain_openai import ChatOpenAI
import os
import load_dotenv

# This function will load all the variables from the .env file and 
# make them available in the os.environ dictionary (env variables)
load_dotenv.load_dotenv() 

if os.environ.get("OPENAI_API_KEY"):
    print("API veriable exists")
else:
    raise ValueError("OPENAI_API_KEY not found")

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser,PydanticOutputParser

llm_openai = ChatOpenAI(model='gpt-5-mini', temperature=0)

API veriable exists


In [7]:
from pydantic import  BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]


llm_structure_output = llm_openai.with_structured_output(llm_schema)

demo_chain = llm_structure_output 
demo_chain.invoke("The movie was great")

llm_schema(movie_summary_flag='positive')

In [12]:
result = llm_structure_output.invoke("This movie is good")
result.model_dump()['movie_summary_flag']

'positive'

# **Conditional Chain**

In [8]:
# Task 1 : Prompt

prompt_template = ChatPromptTemplate([
    ('system', 'You are a movie review evaluator'),
    ('human', "Please catergorize the movie review as positive or negative: {input}")
])

In [9]:
# TASK - 2 [LLM]

llm_structure_output = llm_openai.with_structured_output(llm_schema)


In [14]:
# TASK - 3 : Custom Runnable
from langchain_core.runnables import RunnableLambda

def pydentic_json(input:llm_schema) -> str:
    return input.model_dump()['movie_summary_flag']

pydentic_json_lambda = RunnableLambda(pydentic_json)



# **Chain 1**

In [15]:
# TASK - 1 [PROMPT]
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a linkedin post genrator"),
    ("human", "Create a post for the following text for Linedin: {text}") # This accept dictinarys
])

# TASK - 2 [LLM]
llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

# TASK - 3 [Str Parser]
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_openai | str_parser


### **CHAIN 2**

In [18]:
from langchain_core.runnables import  RunnableParallel, RunnableLambda, RunnableBranch

In [17]:
def insta_chain(text:dict):
    text = text['text']
    # TASK - 1 [PROMPT]
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a e post genrator"),
        ("human", "Create a post for the following text for Instagram: {text}") # This accept dictinarys
    ])

    
    # TASK - 2 [LLM]
    llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

    # TASK - 3 [Str Parser]
    str_parser = StrOutputParser()

    insta_chain = insta_prompt | llm_openai | str_parser

    result = insta_chain.invoke(text)
    return result


insta_chain_runnable = RunnableLambda(insta_chain)



    

### **Final Orchestration**

In [20]:
from email.policy import default

conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linkedin),
    insta_chain_runnable
    
    
)

final_orchestor = prompt_template | llm_structure_output | pydentic_json_lambda | conditional_chain


In [21]:
final_orchestor.invoke({"input": "I liked this moview WOW "})

'Positivity is more than a mood — it\'s a habit that changes outcomes.\n\nThis week I focused on three simple things that made a big difference:\n- Celebrating small wins with the team\n- Reframing setbacks as learning opportunities\n- Saying "thank you" more often\n\nWhen we choose curiosity over blame and appreciation over criticism, collaboration improves and momentum follows. Small shifts in attitude create a ripple effect across culture and performance.\n\nWhat’s one positive moment you experienced this week? Share it below — let’s spread the momentum. \n\n#positivity #leadership #growth #teamwork #wellbeing'